# Gap fill

Upload this whole folder into JupyterLab, open `train_gapfill.ipynb`, and run the cells from top to bottom.

The run downloads KLGX NEXRAD volumes plus ±30-minute temporal context and the Quillayute radiosonde, trains for 20 epochs, and writes `outputs/gapfill_holdout_gaps.png` inside this folder. A CPU kernel is enough. The kernel needs outbound internet.


In [ ]:
from pathlib import Path
import os
import sys

def find_bundle() -> Path:
    """Find this folder even when Jupyter started in the home directory."""
    def ok(path: Path) -> bool:
        return (path / "train_gapfill.ipynb").is_file() and (path / "src" / "radar" / "holdout_gaps.py").is_file()

    cwd = Path.cwd().resolve()
    seeds = [cwd, *cwd.parents]
    home = Path.home().resolve()
    if home not in seeds:
        seeds.append(home)

    seen = set()
    for seed in seeds:
        if seed in seen or not seed.is_dir():
            continue
        seen.add(seed)
        if ok(seed):
            return seed
        try:
            children = list(seed.iterdir())
        except OSError:
            continue
        for child in children:
            if child.is_dir() and ok(child):
                return child
    raise FileNotFoundError(
        "Could not find this folder. Open train_gapfill.ipynb from the folder you uploaded."
    )

ROOT = find_bundle()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("AWS_EC2_METADATA_DISABLED", "true")
print(ROOT)


In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    ("pyart", "arm-pyart"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("numpy", "numpy"),
    ("scipy", "scipy"),
    ("s3fs", "s3fs"),
    ("boto3", "boto3"),
]
missing = [dist for module, dist in packages if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("dependencies already installed")


In [ ]:
from IPython.display import Image, display

from radar.holdout_gaps import run

result = run()
print("middle tilt", result["middle_metrics"])
print("lowest sweep", result["lowest_metrics"])
display(Image(filename=str(result["output_path"])))
